In [4]:

class User:
    """Store basic information shared by customers and employees."""

    def __init__(self, user_id: int, full_name: str, phone_number: str):
        """Initialize a user.

        Args:
            user_id (int): Unique ID of the user.
            full_name (str): Full name of the user.
            phone_number (str): User's phone number.
        """
        self.user_id = user_id
        self.full_name = full_name
        self.phone_number = phone_number

    def enter_user_details(self):
        """Collect and update the user's personal information."""
        try:
            user_id = int(input("Enter your user_id: "))
            full_name = input("Enter your full name: ")
            phone_number = input("Enter your phone number: ")
    
        except ValueError:
            return "Invalid user ID. Please enter a number."
    
        self.user_id = user_id
        self.full_name = full_name
        self.phone_number = phone_number
    

    def display_details(self):
        """Return the user's personal information.

        Returns:
            str: A message containing the user's details.
        """
        return f"The user's name is {self.full_name}, user ID is {self.user_id}, and phone number is {self.phone_number}"


class LoanApplication:
    """Represent a loan application and manage its progress."""

    def __init__(self, application_id, customer_id, loan_amount,
                 duration_in_months, purpose):
        """Create a new loan application.

        Args:
            application_id (int): Unique loan application ID.
            customer_id (int): ID of the customer applying.
            loan_amount (float): Amount requested.
            duration_in_months (int): Loan duration in months.
            purpose (str): Reason for requesting the loan.
        """
        self.application_id = application_id
        self.customer_id = customer_id
        self.loan_amount = loan_amount
        self.duration_in_months = duration_in_months
        self.purpose = purpose

        # Every new loan starts as pending.
        self.status = 'PENDING'

        # Compliance has not verified the application yet.
        self.compliance_passed = False
        self.compliance_notes = None

        # No approval or rejection decision has been made.
        self.decision_reason = None

        # No money has been disbursed or repaid yet.
        self.amount_repaid = 0.0
        self.outstanding_balance = 0.0

    def display_details(self):
        """Return all information about the loan.

        Returns:
            dict: Loan details, status, and repayment information.
        """
        return {
            'application_id': self.application_id,
            'customer_id': self.customer_id,
            'loan_amount': self.loan_amount,
            'duration_in_months': self.duration_in_months,
            'purpose': self.purpose,
            'status': self.status,
            'compliance_passed': self.compliance_passed,
            'compliance_notes': self.compliance_notes,
            'decision_reason': self.decision_reason,
            'amount_repaid': self.amount_repaid,
            'outstanding_balance': self.outstanding_balance
        }

    def approve(self):
        """Approve a pending loan that has passed compliance.

        Returns:
            str: Approval result or an error message.
        """
        # A loan cannot be approved twice or after rejection.
        if self.status != 'PENDING':
            return 'Only pending applications can be approved'

        # Compliance must approve verification before the manager can approve.
        if not self.compliance_passed:
            return 'Application has not passed compliance verification'

        self.status = 'APPROVED'
        return 'Loan application approved successfully'

    def reject(self, reason):
        """Reject a pending loan application.

        Args:
            reason (str): Explanation for the rejection.

        Returns:
            str: Rejection result or an error message.
        """
        if self.status != 'PENDING':
            return 'Only pending applications can be rejected'

        if not reason.strip():
            return 'Rejection reason cannot be empty'

        # Save the rejection reason for future reference.
        self.status = 'REJECTED'
        self.decision_reason = reason

        return 'Loan application rejected'

    def disburse(self):
        """Disburse an approved loan to the customer.

        Returns:
            str: Disbursement result or an error message.
        """
        # Only approved loans can be disbursed.
        if self.status != 'APPROVED':
            return 'Only approved loans can be disbursed'

        self.status = 'DISBURSED'

        # The customer now owes the amount disbursed.
        self.outstanding_balance = float(self.loan_amount)

        return 'Loan disbursed successfully'

    def record_repayment(self, repayment_amount):
        """Record a payment made toward a disbursed loan.

        Args:
            repayment_amount (float): Amount the customer is paying.

        Returns:
            str: Remaining balance or repayment result.
        """
        try:
            repayment_amount = float(repayment_amount)

        except (ValueError, TypeError):
            return "Invalid repayment amount. Enter a valid number."
            
        # Repayment is only allowed after disbursement.
        if self.status != 'DISBURSED':
            return 'This loan is not available for repayment'

        if repayment_amount <= 0:
            return 'Repayment amount must be greater than zero'

        if repayment_amount > self.outstanding_balance:
            return 'Repayment exceeds the outstanding balance'

        # Increase total repayment and reduce the amount owed.
        self.amount_repaid += repayment_amount
        self.outstanding_balance = round(
            self.outstanding_balance - repayment_amount, 2
        )

        # Close the loan when nothing is left to pay.
        if self.outstanding_balance == 0:
            self.status = 'PAID'
            return f'Your loan is fully paid, and loan ID {self.application_id} is closed'

        return f'Your remaining loan balance is {self.outstanding_balance}'


class Customer(User):
    """Represent a customer who can apply for and repay loans."""

    def __init__(self, user_id: int, full_name: str,
                 phone_number: str, monthly_income: float):
        """Initialize a customer.

        Args:
            user_id (int): Customer's unique ID.
            full_name (str): Customer's full name.
            phone_number (str): Customer's phone number.
            monthly_income (float): Customer's monthly income.
        """
        # Get common user details from the User class.
        super().__init__(user_id, full_name, phone_number)

        self.monthly_income = monthly_income

        # Store all loans belonging to this customer.
        self.loan_applications = []

    def apply_for_loan(self, application_id: int, loan_amount: float,
                       duration_in_months: int, purpose: str):
        """Create a loan application after checking the information.

        Args:
            application_id (int): Unique application ID.
            loan_amount (float): Amount requested.
            duration_in_months (int): Loan duration.
            purpose (str): Reason for the loan.

        Returns:
            dict or str: New application details or an error message.
        """
        # Check the customer's income and requested loan details.
        if self.monthly_income <= 0:
            return 'Monthly income must be greater than zero'

        if loan_amount <= 0:
            return 'Loan amount must be greater than zero'

        if duration_in_months <= 0:
            return 'Loan duration must be greater than zero'

        if not purpose.strip():
            return 'Loan purpose cannot be empty'

        # Check all applications so two customers cannot use the same ID.
        for loan in all_applications:
            if loan.application_id == application_id:
                return 'Application ID already exists'

        # Create a new loan using the LoanApplication class.
        loan_application = LoanApplication(
            application_id,
            self.user_id,
            loan_amount,
            duration_in_months,
            purpose
        )

        # Save the same loan object in both lists.
        self.loan_applications.append(loan_application)
        all_applications.append(loan_application)

        return loan_application.display_details()

    def view_my_loans(self):
        """Return all loans belonging to this customer.

        Returns:
            list: List of the customer's loan details.
        """
        return [loan.display_details() for loan in self.loan_applications]

    def make_repayment(self, loan_id, repayment_amount):
        """Make a repayment on one of the customer's loans.

        Args:
            loan_id (int): ID of the loan being repaid.
            repayment_amount (float): Amount being paid.

        Returns:
            str: Repayment result or an error message.
        """
        # Search only loans belonging to this customer.
        for loan in self.loan_applications:
            if loan.application_id == loan_id:
                return loan.record_repayment(repayment_amount)

        return f'Loan ID {loan_id} not found'


class Employee(User):
    """Represent an employee working in the bank."""

    def __init__(self, user_id: int, full_name: str,
                 phone_number: str, employee_id: int, department: str):
        """Initialize an employee.

        Args:
            user_id (int): User ID.
            full_name (str): Employee's name.
            phone_number (str): Employee's phone number.
            employee_id (int): Employee's staff ID.
            department (str): Department where the employee works.
        """
        # Reuse the User constructor instead of repeating its code.
        super().__init__(user_id, full_name, phone_number)

        self.employee_id = employee_id
        self.department = department

    def display_employee_details(self):
        """Return the employee's information.

        Returns:
            dict: Personal and employment details.
        """
        return {
            'user_id': self.user_id,
            'name': self.full_name,
            'phone_number': self.phone_number,
            'employee_id': self.employee_id,
            'department': self.department
        }


class LoanOfficer(Employee):
    """Represent an employee who registers loan applications."""

    def __init__(self, user_id: int, full_name: str,
                 phone_number: str, employee_id: int, department: str,
                 staff_level: str, assigned_region: str):
        """Initialize a loan officer.

        Args:
            user_id (int): User ID.
            full_name (str): Loan officer's name.
            phone_number (str): Phone number.
            employee_id (int): Employee ID.
            department (str): Department name.
            staff_level (str): Staff level.
            assigned_region (str): Region assigned to the officer.
        """
        super().__init__(
            user_id, full_name, phone_number, employee_id, department
        )

        self.staff_level = staff_level
        self.assigned_region = assigned_region

    def register_application(self, customer, application_id,
                             amount, purpose, duration_months):
        """Register a customer's loan application.

        Args:
            customer (Customer): Customer requesting the loan.
            application_id (int): Application ID.
            amount (float): Loan amount.
            purpose (str): Loan purpose.
            duration_months (int): Duration of the loan.

        Returns:
            dict or str: Application details or an error message.
        """
        # Reuse the Customer method so validation is not repeated.
        return customer.apply_for_loan(
            application_id,
            amount,
            duration_months,
            purpose
        )

    def view_pending_applications(self, customers):
        """Find all pending applications from the customers.

        Args:
            customers (list): List of Customer objects.

        Returns:
            list: Details of all pending applications.
        """
        pending_applications = []

        # Check each customer's loans.
        for customer in customers:
            for loan in customer.loan_applications:
                if loan.status == 'PENDING':
                    pending_applications.append(loan.display_details())

        return pending_applications


class BankManager(Employee):
    """Represent a manager who approves or rejects loan applications."""

    def __init__(self, user_id: int, full_name: str,
                 phone_number: str, employee_id: int, department: str):
        """Initialize a bank manager.

        Args:
            user_id (int): User ID.
            full_name (str): Manager's name.
            phone_number (str): Phone number.
            employee_id (int): Employee ID.
            department (str): Department name.
        """
        super().__init__(
            user_id, full_name, phone_number, employee_id, department
        )

    def approve_application(self, application_id):
        """Find and approve a loan application.

        Args:
            application_id (int): ID of the application.

        Returns:
            str: Approval result or an error message.
        """
        # Find the application using its ID.
        for loan in all_applications:
            if loan.application_id == application_id:
                return loan.approve()

        return 'Application ID not found'

    def reject_application(self, application_id, reason):
        """Find and reject a loan application.

        Args:
            application_id (int): ID of the application.
            reason (str): Reason for rejection.

        Returns:
            str: Rejection result or an error message.
        """
        for loan in all_applications:
            if loan.application_id == application_id:
                return loan.reject(reason)

        return 'Application ID not found'


class ComplianceOfficer(Employee):
    """Represent an employee who verifies loan applications."""

    def verify_application(self, application_id, passed, notes):
        """Record the compliance verification result.

        Args:
            application_id (int): ID of the application.
            passed (bool): True if verification passed, otherwise False.
            notes (str): Notes about the verification.

        Returns:
            str: Verification result or an error message.
        """
        for loan in all_applications:
            if loan.application_id == application_id:

                # Only pending applications can be verified.
                if loan.status != 'PENDING':
                    return 'Only pending applications can be verified'

                # Save the verification result and notes.
                loan.compliance_passed = passed
                loan.compliance_notes = notes

                if passed:
                    return 'Loan compliance verification passed'

                return 'Loan compliance verification failed'

        return 'Application ID not found'


class OperationsLead(BankManager, ComplianceOfficer):
    """Represent an employee with both manager and compliance duties."""
    pass


In [5]:

def main():
    """Run the Loan Management System menu."""

    # Store customers and loan applications.
    global all_applications
    all_applications = []
    customers = []

    # Create employees who will process loan applications.
    loan_officer = LoanOfficer(
        201, "Loan Officer", "6051111111",
        301, "Loans", "Senior", "North"
    )

    bank_manager = BankManager(
        202, "Bank Manager", "6052222222",
        302, "Management"
    )

    compliance_officer = ComplianceOfficer(
        203, "Compliance Officer", "6053333333",
        303, "Compliance"
    )

    while True:
        print("\nLOAN MANAGEMENT SYSTEM")
        print("1. Register Customer")
        print("2. Apply for Loan")
        print("3. View Pending Applications")
        print("4. Verify Application")
        print("5. Approve Application")
        print("6. Reject Application")
        print("7. Disburse Loan")
        print("8. Make Repayment")
        print("9. View Customer Loans")
        print("10. View Application Details")
        print("0. Exit")

        choice = input("Enter your choice: ")

        if choice == "0":
            print("Thank you for using the Loan Management System.")
            break

        elif choice == "1":
            try:
                user_id = int(input("Enter customer ID: "))
                name = input("Enter full name: ")
                phone = input("Enter phone number: ")
                income = float(input("Enter monthly income: "))

                if income <= 0:
                    print("Monthly income must be greater than zero")
                    continue

                if any(c.user_id == user_id for c in customers):
                    print("Customer ID already exists")
                    continue

                customer = Customer(user_id, name, phone, income)
                customers.append(customer)

                print("Customer registered successfully")

            except ValueError:
                print("Invalid input. Please enter valid numbers.")

        elif choice == "2":
            try:
                customer_id = int(input("Enter customer ID: "))

                customer = None
                for person in customers:
                    if person.user_id == customer_id:
                        customer = person
                        break

                if customer is None:
                    print("Customer not found")
                    continue

                application_id = int(input("Enter application ID: "))
                amount = float(input("Enter loan amount: "))
                duration = int(input("Enter duration in months: "))
                purpose = input("Enter loan purpose: ")

                print(loan_officer.register_application(
                    customer, application_id, amount, purpose, duration
                ))

            except ValueError:
                print("Invalid loan details. Please enter valid numbers.")

        elif choice == "3":
            pending = loan_officer.view_pending_applications(customers)

            if not pending:
                print("No pending applications")
            else:
                for loan in pending:
                    print(loan)

        elif choice == "4":
            try:
                application_id = int(input("Enter application ID: "))
                answer = input("Did compliance pass? (yes/no): ").lower()
                notes = input("Enter verification notes: ")

                if answer not in ("yes", "no"):
                    print("Please enter yes or no")
                    continue

                print(compliance_officer.verify_application(
                    application_id, answer == "yes", notes
                ))

            except ValueError:
                print("Invalid application ID.")

        elif choice == "5":
            try:
                application_id = int(input("Enter application ID: "))
                print(bank_manager.approve_application(application_id))
            except ValueError:
                print("Invalid application ID.")

        elif choice == "6":
            try:
                application_id = int(input("Enter application ID: "))
                reason = input("Enter rejection reason: ")

                print(bank_manager.reject_application(
                    application_id, reason
                ))

            except ValueError:
                print("Invalid application ID.")

        elif choice == "7":
            try:
                application_id = int(input("Enter application ID: "))

                for loan in all_applications:
                    if loan.application_id == application_id:
                        print(loan.disburse())
                        break
                else:
                    print("Application not found")

            except ValueError:
                print("Invalid application ID.")

        elif choice == "8":
            try:
                customer_id = int(input("Enter customer ID: "))
                application_id = int(input("Enter loan ID: "))
                amount = float(input("Enter repayment amount: "))

                for customer in customers:
                    if customer.user_id == customer_id:
                        print(customer.make_repayment(application_id, amount))
                        break
                else:
                    print("Customer not found")

            except ValueError:
                print("Invalid repayment details.")

        elif choice == "9":
            try:
                customer_id = int(input("Enter customer ID: "))

                for customer in customers:
                    if customer.user_id == customer_id:
                        print(customer.view_my_loans())
                        break
                else:
                    print("Customer not found")

            except ValueError:
                print("Invalid customer ID.")

        elif choice == "10":
            try:
                application_id = int(input("Enter application ID: "))

                for loan in all_applications:
                    if loan.application_id == application_id:
                        print(loan.display_details())
                        break
                else:
                    print("Application not found")

            except ValueError:
                print("Invalid application ID.")

        else:
            print("Invalid menu option. Please try again.")


In [6]:
if __name__ == "__main__":
    main()


LOAN MANAGEMENT SYSTEM
1. Register Customer
2. Apply for Loan
3. View Pending Applications
4. Verify Application
5. Approve Application
6. Reject Application
7. Disburse Loan
8. Make Repayment
9. View Customer Loans
10. View Application Details
0. Exit


Enter your choice:  1
Enter customer ID:  101
Enter full name:  joy okolo
Enter phone number:  +2348134237155
Enter monthly income:  1000000


Customer registered successfully

LOAN MANAGEMENT SYSTEM
1. Register Customer
2. Apply for Loan
3. View Pending Applications
4. Verify Application
5. Approve Application
6. Reject Application
7. Disburse Loan
8. Make Repayment
9. View Customer Loans
10. View Application Details
0. Exit


Enter your choice:  2
Enter customer ID:  101
Enter application ID:  101
Enter loan amount:  5000000
Enter duration in months:  6
Enter loan purpose:  tuition fee


{'application_id': 101, 'customer_id': 101, 'loan_amount': 5000000.0, 'duration_in_months': 6, 'purpose': 'tuition fee', 'status': 'PENDING', 'compliance_passed': False, 'compliance_notes': None, 'decision_reason': None, 'amount_repaid': 0.0, 'outstanding_balance': 0.0}

LOAN MANAGEMENT SYSTEM
1. Register Customer
2. Apply for Loan
3. View Pending Applications
4. Verify Application
5. Approve Application
6. Reject Application
7. Disburse Loan
8. Make Repayment
9. View Customer Loans
10. View Application Details
0. Exit


Enter your choice:  Exit


Invalid menu option. Please try again.

LOAN MANAGEMENT SYSTEM
1. Register Customer
2. Apply for Loan
3. View Pending Applications
4. Verify Application
5. Approve Application
6. Reject Application
7. Disburse Loan
8. Make Repayment
9. View Customer Loans
10. View Application Details
0. Exit


Enter your choice:  0


Thank you for using the Loan Management System.


In [12]:
# Create an empty list for all loan applications.
all_applications = []

# Create an empty list for customers.
customers = []

# Create two customer objects.
customer1 = Customer(101, "Joy", "6051234567", 5000)
customer2 = Customer(102, "John", "6059876543", 3500)

# Add both customers to the customers list.
customers.append(customer1)
customers.append(customer2)

# Create a loan officer object.
loan_officer = LoanOfficer(
    201, "Loan Officer", "6051111111",
    301, "Loans", "Senior", "North"
)

print("Test objects created successfully")

Test objects created successfully


In [13]:

# Test a valid loan application.
print(customer1.apply_for_loan(1001, 2000, 12, "Business"))

# Test another customer's application through the loan officer.
print(loan_officer.register_application(
    customer2, 1002, 1500, "Education", 6
))

# Test an invalid loan amount.
print(customer1.apply_for_loan(1003, -500, 12, "Business"))

# Test an invalid duration.
print(customer1.apply_for_loan(1004, 1000, 0, "Education"))

# Test a duplicate application ID.
print(customer2.apply_for_loan(1001, 1000, 12, "Business"))


{'application_id': 1001, 'customer_id': 101, 'loan_amount': 2000, 'duration_in_months': 12, 'purpose': 'Business', 'status': 'PENDING', 'compliance_passed': False, 'compliance_notes': None, 'decision_reason': None, 'amount_repaid': 0.0, 'outstanding_balance': 0.0}
{'application_id': 1002, 'customer_id': 102, 'loan_amount': 1500, 'duration_in_months': 6, 'purpose': 'Education', 'status': 'PENDING', 'compliance_passed': False, 'compliance_notes': None, 'decision_reason': None, 'amount_repaid': 0.0, 'outstanding_balance': 0.0}
Loan amount must be greater than zero
Loan duration must be greater than zero
Application ID already exists


In [14]:

# Display all loan applications that are still pending.
pending_loans = loan_officer.view_pending_applications(customers)

for loan in pending_loans:
    print(loan)


{'application_id': 1001, 'customer_id': 101, 'loan_amount': 2000, 'duration_in_months': 12, 'purpose': 'Business', 'status': 'PENDING', 'compliance_passed': False, 'compliance_notes': None, 'decision_reason': None, 'amount_repaid': 0.0, 'outstanding_balance': 0.0}
{'application_id': 1002, 'customer_id': 102, 'loan_amount': 1500, 'duration_in_months': 6, 'purpose': 'Education', 'status': 'PENDING', 'compliance_passed': False, 'compliance_notes': None, 'decision_reason': None, 'amount_repaid': 0.0, 'outstanding_balance': 0.0}


In [25]:

# Create the bank manager object.
bank_manager = BankManager(
    202, "Bank Manager", "6052222222",
    302, "Management"
)

# Create the compliance officer object.
compliance_officer = ComplianceOfficer(
    203, "Compliance Officer", "6053333333",
    303, "Compliance"
)

# Try approving before compliance verification.
print(bank_manager.approve_application(1001))

# Verify the first loan successfully.
print(compliance_officer.verify_application(
    1001, True, "Customer identity and income verified"
))

# Record a failed verification for the second loan.
print(compliance_officer.verify_application(
    1002, False, "Income could not be verified"
))


Loan application approved successfully
Only pending applications can be verified
Loan compliance verification failed


In [26]:

# Approve the first loan after successful compliance verification.
print(bank_manager.approve_application(1001))

# Attempt to approve a loan that failed compliance.
print(bank_manager.approve_application(1002))

# Reject the second loan and save the reason.
print(bank_manager.reject_application(
    1002, "Customer did not pass compliance verification"
))


Only pending applications can be approved
Application has not passed compliance verification
Loan application rejected


In [16]:

for loan in all_applications:
    print(
        "Application ID:", loan.application_id,
        "| Compliance:", loan.compliance_passed,
        "| Status:", loan.status
    )

Application ID: 1001 | Compliance: False | Status: PENDING
Application ID: 1002 | Compliance: False | Status: PENDING


In [27]:

# Make the first repayment.
print(customer1.make_repayment(1001, 500))

# Make another repayment.
print(customer1.make_repayment(1001, 1000))

# Pay the remaining balance.
print(customer1.make_repayment(1001, 500))

# Check that the loan is now fully paid.
print(customer1.view_my_loans())


This loan is not available for repayment
This loan is not available for repayment
This loan is not available for repayment
[{'application_id': 1001, 'customer_id': 101, 'loan_amount': 2000, 'duration_in_months': 12, 'purpose': 'Business', 'status': 'APPROVED', 'compliance_passed': True, 'compliance_notes': 'Customer identity and income verified', 'decision_reason': None, 'amount_repaid': 0.0, 'outstanding_balance': 0.0}]


In [29]:

operations_lead = OperationsLead(
    204, "Operations Lead", "6054444444",
    304, "Operations"
)
# Confirm that OperationsLead inherits both methods.
print(hasattr(operations_lead, "approve_application"))
print(hasattr(operations_lead, "verify_application"))

# Show the employee's details inherited from Employee.
print(operations_lead.display_employee_details())

# Show the inheritance order.
print(OperationsLead.__mro__)


True
True
{'user_id': 204, 'name': 'Operations Lead', 'phone_number': '6054444444', 'employee_id': 304, 'department': 'Operations'}
(<class '__main__.OperationsLead'>, <class '__main__.BankManager'>, <class '__main__.ComplianceOfficer'>, <class '__main__.Employee'>, <class '__main__.User'>, <class 'object'>)
